# Ocean meridional heat transport

Two ways to get the ocean's meridional heat transport, as in the
original notebook.

**Method 1, the online diagnostic.** MOM5 integrates the resolved
advective heat flux online into `temp_yflux_adv_int_z`. The CMIP6
equivalent is `hfbasin`, which is already zonally integrated, so
there is nothing to do but convert W to PW.

**Method 2, the surface flux integration.** Where `hfbasin` is not
published, integrate the net surface heat flux `hfds` northward
from the south pole. This assumes a steady state; the notebook's
warning about that applies here unchanged.

Note that the surface-flux method deliberately does **not**
regrid: the latitude binning runs on the model's own grid so that
the cell areas and the flux stay consistent.

---

Translated from the COSIMA recipe
[`Meridional_heat_transport.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Meridional_heat_transport.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
from esmvalcore.preprocessor import climate_statistics

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', mip='Omon', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              timerange='1990/2009')

hfds = Dataset(short_name='hfds', dataset='ACCESS-CM2', **common)
hfds.add_supplementary(short_name='areacello', mip='Ofx')

hfbasin = Dataset(short_name='hfbasin', dataset='ACCESS-CM2',
                  **common)

print('hfds files:', list(hfds.files))

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def time_mean(cube):
    """Average over time if the cube still carries a time coordinate."""
    if cube.coords("time") and cube.coord("time").shape[0] > 1:
        return cube.collapsed("time", iris.analysis.MEAN)
    if cube.coords("time"):
        return iris.util.squeeze(cube)
    return cube


def bin_centres(bins):
    """Midpoints of a monotonic array of bin edges."""
    bins = np.asarray(bins, dtype=float)
    return 0.5 * (bins[1:] + bins[:-1])


def bin_sum(values, bin_variable, bins, axis=None):
    """Sum ``values`` into ``bins`` of ``bin_variable``.

    ``values`` and ``bin_variable`` must have the same shape.  Returns
    an array whose first axis runs over the bin centres and whose
    remaining axes are the axes of ``values`` *not* listed in ``axis``.
    With ``axis=None`` everything is collapsed into a 1D profile.

    This is ``xhistogram.xarray.histogram(bin_variable, bins=bins,
    weights=values, dim=axis)`` without the xarray dependency.
    """
    values = np.ma.masked_invalid(np.ma.asarray(values, dtype=float))
    bin_variable = np.ma.masked_invalid(
        np.ma.asarray(bin_variable, dtype=float))
    bins = np.asarray(bins, dtype=float)
    n_bins = bins.size - 1

    if axis is None:
        axis = tuple(range(values.ndim))
    elif isinstance(axis, int):
        axis = (axis,)
    axis = tuple(a % values.ndim for a in axis)
    keep = tuple(a for a in range(values.ndim) if a not in axis)

    moved = np.ma.transpose(values, keep + axis)
    moved_bin = np.ma.transpose(bin_variable, keep + axis)
    kept_shape = moved.shape[:len(keep)]
    flat = moved.reshape(int(np.prod(kept_shape)) if keep else 1, -1)
    flat_bin = moved_bin.reshape(flat.shape)

    out = np.zeros((n_bins,) + kept_shape)
    flat_out = out.reshape(n_bins, -1)
    # np.digitize returns 0 below the first edge and n_bins+1 above the
    # last, so a bin index of 1..n_bins is inside the range.
    for column in range(flat.shape[0]):
        data = flat[column]
        coordinate = flat_bin[column]
        valid = ~np.ma.getmaskarray(data) & ~np.ma.getmaskarray(coordinate)
        if not valid.any():
            continue
        index = np.digitize(
            np.ma.filled(coordinate, np.nan)[valid], bins)
        inside = (index >= 1) & (index <= n_bins)
        flat_out[:, column] = np.bincount(
            index[inside] - 1,
            weights=np.ma.filled(data, 0.0)[valid][inside],
            minlength=n_bins)
    return out


def latitude_bin_edges(cube, n_bins=None):
    """Bin edges spanning the cube's *whole* latitude field.

    Do not build these from :func:`lat_1d`.  On a CMORised tripolar grid
    there is no 1D nominal latitude to fall back on -- ``latitude`` is a
    2D auxiliary coordinate -- and ``lat_1d`` returns one arbitrary
    column of it.  North of the tripolar fold the grid rows stop
    following latitude circles, so that column's maximum depends on
    which meridian it happens to sit on: for a displaced-pole Arctic it
    can stop dead at the fold (~65N) while the field itself reaches
    89.5N.  Edges built from it silently discard every cell beyond,
    which is the entire Arctic.

    The upstream COSIMA notebook does not hit this because MOM5 output
    carries ``yt_ocean``, a genuine 1D nominal latitude spanning the
    full grid, alongside the 2D ``geolat_t``.  CMOR keeps only the 2D
    field, so the edges have to be constructed instead of borrowed.

    Returns ``n_bins + 1`` edges from -90 to 90, defaulting to one bin
    per grid row.
    """
    lat_field = lat_2d(cube)
    if n_bins is None:
        n_bins = lat_field.shape[0]
    return np.linspace(-90.0, 90.0, int(n_bins) + 1)


def latitude_bin_sum(values, lat_field, lat_bins):
    """Sum ``values`` into latitude bins of a possibly curvilinear grid.

    This is the ``groupby_bins('geolat_t', ...)`` of
    ``Meridional_heat_transport.ipynb``.  On a regular grid it reduces
    to a zonal sum.
    """
    lat_field = np.broadcast_to(np.asarray(lat_field), np.shape(values))
    return bin_sum(values, lat_field, lat_bins)


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)

### Where the latitude bins come from

The original writes its bins as
`np.hstack(([-90], area['yt_ocean'].values))` — the model's own
nominal latitude. That does not survive CMORisation, and the
difference is not cosmetic.

MOM5 output carries *two* latitudes: `yt_ocean`, a 1-D nominal
coordinate spanning the whole grid, and `geolat_t`, the true 2-D
latitude. The original bins the second by the first. **CMOR keeps
only the 2-D field** — on a `gn` tripolar grid `latitude` is an
auxiliary coordinate and the `j` dimension is a bare index — so
there is no 1-D latitude left to borrow.

Taking one column of the 2-D field instead is the trap. North of
the tripolar fold the rows stop following latitude circles, so a
column's maximum depends on which meridian it sits on: for a
displaced-pole Arctic, column 0 can stop dead at **65°N** while
the field itself reaches 89.5°N. Bins built from it discard every
cell beyond — the entire Arctic — and say nothing.

The symptom is a transport curve that simply ends at 65°N. If you
see that, this is why.

So `latitude_bin_edges` constructs the bins to span −90 to 90
with one bin per grid row, and the cell below checks that no
ocean cell falls outside them.

## Why this one is not regridded

Every other notebook here calls `regrid` to get onto a tidy
regular grid. This one deliberately does not, and that choice
forces a second one.

The COSIMA original bins the surface flux by `geolat_t`, the
model's own 2-D latitude, and weights by `area_t`. That is not a
stylistic detail: the integral
$\int Q\,\mathrm{d}A$ is only correct if the flux and the area
refer to the *same* cells. Regridding first would interpolate the
flux onto cells whose areas we would then have to recompute, and
the ocean's coastline would be smeared in the process.

Staying on the native grid costs us something, though. Almost
every CMIP6 ocean is **tripolar**, so `latitude` is a 2-D
auxiliary coordinate, not a 1-D axis. Cell areas therefore cannot
come from a "representative" 1-D latitude — that is meaningless
where cells do not follow latitude circles — and iris's
`area_weights` simply raises `CoordinateMultiDimError`.
ESMValCore gives up at the same point: its
`try_adding_calculated_cell_area` handles regular and
rotated-pole grids and raises for irregular ones.

### So where does the area come from?

`cell_area` below tries three sources in order.

**1. `areacello`** — the model's own areas, and always the best
answer, because it reflects how the grid was actually generated.
That is what the `add_supplementary` call above requests. If the
experiment you want does not publish it, borrow it from another
experiment of the same model; the ocean grid does not change
between them:

```python
hfds.add_supplementary(short_name='areacello', mip='Ofx',
                       exp='piControl')
```

**2. The cell corner vertices** — the part worth knowing, because
it means a missing `areacello` is *not* fatal. CMOR mandates
bounds on `latitude` and `longitude`, and on a curvilinear grid
those bounds are the four corners of every cell. The area of the
spherical polygon they define follows exactly, from the spherical
excess. No supplementary file is involved: the information is
already in the data file.

**3. 1-D coordinate bounds** — regular grids only.

Route 2 assumes great-circle cell edges, which differs from a
latitude-longitude box area by about 0.02% on a 2-degree cell.
That is far below the difference between either and `areacello`,
which is why `areacello` still wins when it is there.

## Method 2: integrate the surface heat flux

$\mathrm{MHF}(y) = \int_{-90}^{y} Q \, \mathrm{d}A$

accumulated over the model's own latitude bins, then centred by
$(\mathrm{MHF}[0] - \mathrm{MHF}[-1])/2$ so the residual of the
steady-state assumption is split between the two poles.

In [ ]:
WATTS_TO_PW = 1.0e-15

cube = climate_statistics(hfds.load(), operator='mean',
                          period='full')
flux = masked_data(cube)

# areacello, not a coordinate-derived area: see above. On a
# tripolar grid there is no 1D latitude to derive one from.
area = cell_area(cube)

# Bin by the model's own 2D latitude, which is what makes this
# a zonal integral on a curvilinear grid. The edges span the
# whole field, NOT one column of it -- see above.
lat_field = lat_2d(cube)
edges = latitude_bin_edges(cube)
lat = bin_centres(edges)

# A discarded cell is heat that never enters the integral.
inside = (lat_field >= edges[0]) & (lat_field <= edges[-1])
covered = np.ma.mean(np.ma.masked_where(
    np.ma.getmaskarray(flux), inside).astype(float))
print(f'{100 * covered:.1f}% of ocean cells fall inside the bins'
      f'  (latitude field spans {lat_field.min():.1f} to '
      f'{lat_field.max():.1f} N)')

binned = latitude_bin_sum(flux * area, lat_field, edges)

transport = np.cumsum(binned)
imbalance = transport[-1] * WATTS_TO_PW
transport = transport + (transport[0] - transport[-1]) / 2.0
transport = transport * WATTS_TO_PW

print(f'peak northward transport: {transport.max():.2f} PW')
print(f'net surface heat gain    : {imbalance:+.2f} PW '
      f'(split +-{abs(imbalance)/2:.2f} PW across the ends '
      f'by the centring)')

## Method 1: read `hfbasin`

`hfbasin` may carry a basin dimension; the first one is the global
ocean. Skip this cell if the model does not publish `hfbasin`.

In [ ]:
try:
    basin_cube = climate_statistics(hfbasin.load(),
                                    operator='mean', period='full')
    basin_transport = np.squeeze(
        masked_data(basin_cube)) * WATTS_TO_PW
    while basin_transport.ndim > 1:
        basin_transport = basin_transport[0]
    basin_lat = lat_1d(basin_cube)
except Exception as error:
    print('hfbasin not available:', error)
    basin_transport = None

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(lat, transport, color='green', label='Method 2 (hfds)')
if basin_transport is not None:
    ax.plot(basin_lat, basin_transport, color='blue',
            label='Method 1 (hfbasin)')

ax.axhline(0, color='black', linewidth=1)
ax.set_xlim(-90, 90)
ax.set_xticks(np.arange(-90, 91, 30))
ax.set_xlabel('Latitude')
ax.set_ylabel('Heat transport (PW)')
ax.set_title('Global ocean meridional heat transport')
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()

## Reading the result

Three things to check before believing the curve.

**Does it span the full latitude range?** It should run from the
Antarctic coast to the northern limit of the ocean. A curve that
stops around 65°N means the Arctic was dropped — see the binning
note above.

**Are the magnitudes right?** Observational estimates put the
global peak northward transport at roughly 1.5–2.0 PW near
15–20°N (Trenberth & Caron 2001; Ganachaud & Wunsch 2000), with a
southward peak of about −0.8 to −1.4 PW near 10–15°S. A model
landing a little below the observed peak is common, and Method 2
tends to sit lower than Method 1 anyway because it inherits every
bias in the surface flux field.

**Do the ends go to zero?** They should, because no heat crosses
the northern or southern boundary of a closed ocean. They will
not, and the offset is informative rather than a bug: it is the
net surface heat gain printed above. Method 2 assumes a steady
state, and a `historical` run is warming, so the ocean genuinely
takes up a few tenths of a PW. The centring spreads that
imbalance symmetrically across the two ends — **a presentational
convention, not a physical result**. An offset of order 0.2 PW at
each end is ordinary for a warming run; one of order 1 PW means
something else is wrong, most likely missing cells.

Method 1 (`hfbasin`) has none of this trouble: it is the model's
own online transport, so it needs no steady-state assumption and
no centring. Prefer it wherever the model publishes it, and treat
Method 2 as the fallback the original says it is.

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()